In [1]:
import os

os.chdir(r"C:\Users\1144100924\Documents\ANGIE\PYTHON\Codigo")

In [3]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
from rapidfuzz import fuzz, process


# =============================================================================
# CONFIGURACIÓN
# =============================================================================

CARPETA = Path(r"C:\Users\1144100924\Documents\ANGIE\PYTHON\Codigo")

ARCHIVO_JOVENES = CARPETA / "Para Python.xlsx"
ARCHIVO_MATRICULADOS = CARPETA / "MATRICULADOS_POSGRADO.xlsx"
ARCHIVO_SALIDA = CARPETA / "asistencias_coincidencias.xlsx"

HOJA_JOVENES = "asistentes"
HOJA_MATRICULADOS = "Hoja1"

COL_JOVEN = "Nombre del jóven investigador"
COL_NOMBRE = "Nombre completo"
COL_PROGRAMA = "nombre_programa"

UMBRAL = 86


# =============================================================================
# LIMPIEZA DE NOMBRES
# =============================================================================

def limpiar_nombre(valor):
    if pd.isna(valor):
        return ""

    texto = str(valor).strip().upper()

    # Eliminar tildes
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        caracter
        for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )

    # Eliminar caracteres especiales y espacios duplicados
    texto = re.sub(r"[^A-Z0-9 ]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


# =============================================================================
# VALIDAR ARCHIVOS
# =============================================================================

if not ARCHIVO_JOVENES.exists():
    raise FileNotFoundError(f"No se encontró: {ARCHIVO_JOVENES}")

if not ARCHIVO_MATRICULADOS.exists():
    raise FileNotFoundError(f"No se encontró: {ARCHIVO_MATRICULADOS}")


# =============================================================================
# CARGAR ARCHIVOS
# =============================================================================

# Leer todas las hojas para conservarlas
hojas_jovenes = pd.read_excel(
    ARCHIVO_JOVENES,
    sheet_name=None,
    dtype=object
)

df_jovenes = hojas_jovenes[HOJA_JOVENES].copy()

df_matriculados = pd.read_excel(
    ARCHIVO_MATRICULADOS,
    sheet_name=HOJA_MATRICULADOS,
    dtype=object
)


# Limpiar espacios únicamente en los encabezados
df_jovenes.columns = df_jovenes.columns.astype(str).str.strip()
df_matriculados.columns = df_matriculados.columns.astype(str).str.strip()


# =============================================================================
# VALIDAR COLUMNAS
# =============================================================================

if COL_JOVEN not in df_jovenes.columns:
    raise ValueError(
        f"No existe la columna '{COL_JOVEN}' en Para Python.xlsx.\n"
        f"Columnas disponibles: {list(df_jovenes.columns)}"
    )

for columna in [COL_NOMBRE, COL_PROGRAMA]:
    if columna not in df_matriculados.columns:
        raise ValueError(
            f"No existe la columna '{columna}' "
            f"en MATRICULADOS_POSGRADO.xlsx.\n"
            f"Columnas disponibles: {list(df_matriculados.columns)}"
        )


# =============================================================================
# PREPARAR NOMBRES
# =============================================================================

# Se conserva intacta la columna original:
# Nombre del jóven investigador

# Crear una columna auxiliar limpia para realizar la comparación
df_jovenes["_nombre_limpio_joven"] = (
    df_jovenes[COL_JOVEN].apply(limpiar_nombre)
)

# Crear columna auxiliar limpia en matriculados
df_matriculados["_nombre_limpio"] = (
    df_matriculados[COL_NOMBRE].apply(limpiar_nombre)
)

# Eliminar registros de referencia sin nombre
df_matriculados = df_matriculados[
    df_matriculados["_nombre_limpio"] != ""
].reset_index(drop=True)

nombres_referencia = df_matriculados["_nombre_limpio"].tolist()


# =============================================================================
# BUSCAR COINCIDENCIAS
# =============================================================================

def encontrar_coincidencia(nombre_limpio):
    if not nombre_limpio:
        return pd.Series([pd.NA, pd.NA, 0])

    resultado = process.extractOne(
        nombre_limpio,
        nombres_referencia,
        scorer=fuzz.WRatio,
        score_cutoff=UMBRAL
    )

    if resultado is None:
        return pd.Series([pd.NA, pd.NA, 0])

    _, porcentaje, posicion = resultado

    fila = df_matriculados.iloc[posicion]

    return pd.Series([
        fila[COL_NOMBRE],
        fila[COL_PROGRAMA],
        round(float(porcentaje), 2)
    ])


df_jovenes[
    ["coincidencia", "Programa_fin", "porcentaje_coincidencia"]
] = df_jovenes["_nombre_limpio_joven"].apply(encontrar_coincidencia)


# Eliminar la columna auxiliar antes de guardar
df_jovenes.drop(
    columns=["_nombre_limpio_joven"],
    inplace=True
)


# =============================================================================
# GUARDAR RESULTADO
# =============================================================================

hojas_jovenes[HOJA_JOVENES] = df_jovenes

with pd.ExcelWriter(
    ARCHIVO_SALIDA,
    engine="openpyxl"
) as writer:

    for nombre_hoja, dataframe in hojas_jovenes.items():
        dataframe.to_excel(
            writer,
            sheet_name=nombre_hoja[:31],
            index=False
        )


# =============================================================================
# RESUMEN
# =============================================================================

encontradas = df_jovenes["coincidencia"].notna().sum()

print("Proceso terminado.")
print(f"Registros procesados: {len(df_jovenes)}")
print(f"Coincidencias encontradas: {encontradas}")
print(f"Sin coincidencia: {len(df_jovenes) - encontradas}")
print(f"Archivo generado: {ARCHIVO_SALIDA}")

Proceso terminado.
Registros procesados: 198
Coincidencias encontradas: 0
Sin coincidencia: 198
Archivo generado: C:\Users\1144100924\Documents\ANGIE\PYTHON\Codigo\asistencias_coincidencias.xlsx
